# 06 - Hedonic Regression (Descriptive)

**inputs**  
- `nri_panel_smooth` NRI county-month interpolated  
- `../data/zillow_county_smooth_zhvi.csv` seasonality-adjusted and 3 month smoothed zillow data county-month  

**outputs**  
- `../data/processed/hedonic_dataset.pkl`  
- `../data/processed/hedonic_dataset.csv` analysis set for hedonic regression  

**filters**  
Nov 2020 - Dec 2025
- could add recent months, but the idea is that we don't know where NRI is going post 12/25 so it gets further from the measurement
- could limit to March 2023 - Dec 2025 to use a time period with only 1 Sovi measurement type  


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
print(np.__version__)

2.3.5


## Load NRI and Zillow Data
nri_panel parquet and zillow csv

In [3]:
NRI_TYPE = '_smooth'
ZILLOW_TYPE = 'smooth'
zillow   = pd.read_csv(f'../data/raw/zillow_county_{ZILLOW_TYPE}_zhvi.csv')
nri      = pd.read_pickle(f'../data/processed/nri_panel{NRI_TYPE}.pkl')

print(f'Zillow data:  {zillow.shape}')
print(f'NRI panel:     {nri.shape}')

Zillow data:  (3073, 324)
NRI panel:     (226368, 14)


In [4]:
MIN_DATE = '2020-05-01'
MAX_DATE = '2026-03-31'

# Load full county ZHVI wide format
META_COLS = ['RegionID', 'SizeRank', 'RegionName', 'RegionType',
             'StateName', 'State', 'Metro', 'StateCodeFIPS', 'MunicipalCodeFIPS',
             'latitude', 'longitude']

date_cols = [c for c in zillow.columns if c not in META_COLS]

# Build long format for all dates
zhvi_long = zillow.melt(
    id_vars=['StateCodeFIPS', 'MunicipalCodeFIPS'],
    value_vars=date_cols,
    var_name='date',
    value_name='zhvi'
)
zhvi_long['date'] = pd.to_datetime(zhvi_long['date'])
zhvi_long['state_fips']  = zhvi_long['StateCodeFIPS'].astype(str).str.zfill(2)
zhvi_long['county_fips'] = zhvi_long['MunicipalCodeFIPS'].astype(str).str.zfill(3)
zhvi_long['stcofips']    = zhvi_long['state_fips'] + zhvi_long['county_fips']
zhvi_long['year']        = zhvi_long['date'].dt.year
zhvi_long['month']       = zhvi_long['date'].dt.month
zhvi_long = zhvi_long.dropna(subset=['zhvi'])

# Filter to Nov 2020 - present
zhvi_long = zhvi_long[(zhvi_long['date'] >= MIN_DATE) & (zhvi_long['date'] <= MAX_DATE)]

# Index by stcofips + year + month for fast lookup
zhvi_idx = zhvi_long.set_index(['stcofips', 'year', 'month'])['zhvi']

print(f'ZHVI lookup: {len(zhvi_idx):,} county-month records')
print(f'Date range: {zhvi_long["date"].min().date()} to {zhvi_long["date"].max().date()}')

ZHVI lookup: 216,748 county-month records
Date range: 2020-05-31 to 2026-03-31


## Filter NRI data to Nov 2020 - Dec 2025

In [5]:
# drop storm_year column, rename vintage to year
# 62 months x 3144 counties = 194,928
drop_cols = ['state_fips','county_fips','stateabbrv',]
nri = nri.drop(columns= drop_cols,axis=1)
nri.rename(columns={"storm_year":"year"},inplace=True)
nri = nri[(nri['year'] > 2020) | ((nri['year'] == 2020) & (nri['month'] >= 5))]
nri.shape

(213792, 11)

## Join NRI and Zillow Data

In [6]:
nri['zhvi'] = pd.MultiIndex.from_arrays(
    [nri['stcofips'], nri['year'], nri['month']]
).map(zhvi_idx)

print(nri.shape)
print(nri.isna().sum())
print('Deleting ',nri['zhvi'].isna().sum(),'null zhvi values')
nri = nri.dropna(subset=['zhvi'])
nri.shape

(213792, 12)
stcofips          0
county            0
year              0
month             0
nri_vintage       0
resl_score        0
resl_value        0
risk_value        0
eal_valt          0
sovi_score        0
crf_value         0
zhvi           6875
dtype: int64
Deleting  6875 null zhvi values


(206917, 12)

In [7]:
nri.head()

,stcofips,county,year,month,nri_vintage,resl_score,resl_value,risk_value,eal_valt,sovi_score,crf_value,zhvi
4,01001,Autauga,2020,5,2020,55.5298,2.77649,2.602012e+06,3.311627e+06,25.857312,0.78572,188048.841338
5,01001,Autauga,2020,6,2020,55.5298,2.77649,2.602012e+06,3.311627e+06,25.857312,0.78572,188497.687445
6,01001,Autauga,2020,7,2020,55.5298,2.77649,2.602012e+06,3.311627e+06,25.857312,0.78572,189065.572221
7,01001,Autauga,2020,8,2020,55.5298,2.77649,2.602012e+06,3.311627e+06,25.857312,0.78572,190111.810797
8,01001,Autauga,2020,9,2020,55.5298,2.77649,2.602012e+06,3.311627e+06,25.857312,0.78572,191478.597518


## Cross-section robustness: one observation per county per NRI vintage

Self-contained pipeline. Pulls NRI vintage-level fields directly from `NRI_Long.csv` and
derives the coastal flag from the source NRI tables, so it doesn't depend on
`hedonic_dataset.csv` being current.

**Output:** one row per county per NRI vintage, with **mean ZHVI** over that vintage's coverage window.

Vintage windows +- 6 mo (matching the NRI release-anchor schedule):
- **NRI 2020** anchor → mean ZHVI May 2020 – Apr 2021 (12 months)
- **NRI 2021** anchor → mean ZHVI May 2021 – Apr 2022 (12 months)
- **NRI 2023** anchor → mean ZHVI Sept 2022 – Aug 2023 (12 months)
- **NRI 2025** anchor → mean ZHVI June 2025 – March 2026 (9 months)

Output: `../data/processed/hedonic_xsec.csv`

In [8]:
# Build per-county mean ZHVI for each vintage's window
# zhvi_long was created in cell 4; if you've restarted the kernel, re-run cell 4 first.

VINTAGE_WINDOWS = {
    2020: ('2020-05-01', '2021-04-30'),
    2021: ('2021-05-01', '2022-04-30'),
    2023: ('2022-09-01', '2023-08-31'),
    2025: ('2025-06-01', '2026-03-31')
}

frames = []
for vintage, (start, end) in VINTAGE_WINDOWS.items():
    win = zhvi_long[(zhvi_long['date'] >= start) & (zhvi_long['date'] <= end)]
    means = (
        win.groupby('stcofips')
           .agg(zhvi=('zhvi', 'mean'),
                n_months=('zhvi', 'size'),
                window_start=('date', 'min'),
                window_end=('date', 'max'))
           .reset_index()
    )
    means['nri_vintage'] = vintage
    frames.append(means)

zhvi_xsec = pd.concat(frames, ignore_index=True)
print(f'ZHVI per county-vintage rows: {len(zhvi_xsec):,}')
print(zhvi_xsec.groupby('nri_vintage').agg(
    counties=('stcofips', 'nunique'),
    median_n_months=('n_months', 'median')
))
zhvi_xsec.head()

ZHVI per county-vintage rows: 12,220
             counties  median_n_months
nri_vintage                           
2020             3022             12.0
2021             3061             12.0
2023             3064             12.0
2025             3073             10.0


,stcofips,zhvi,n_months,window_start,window_end,nri_vintage
0,01001,193902.048732,12,2020-05-31,2021-04-30,2020
1,01003,270096.763822,12,2020-05-31,2021-04-30,2020
2,01005,129636.196611,12,2020-05-31,2021-04-30,2020
3,01007,179854.050043,12,2020-05-31,2021-04-30,2020
4,01009,183039.690852,12,2020-05-31,2021-04-30,2020


In [9]:
# Pull vintage-level NRI fields directly from NRI_Long.csv
# (population, area, resl_value are here for all vintages; sovi_value only for 2020/2021)

NRI_LONG_COLS = [
    'nri_id', 'stateabbrv', 'county',
    'population', 'area', 'buildvalue', 'agrivalue',
    'risk_value', 'eal_valt', 'crf_value',
    'sovi_score', 'resl_score', 'resl_value',
    'year'  # NRI vintage
]

nri_long = pd.read_csv('../data/processed/NRI_Long.csv', usecols=NRI_LONG_COLS)
nri_long = nri_long.rename(columns={'year': 'nri_vintage'})
nri_long['stcofips'] = nri_long['nri_id'].str[1:]

# SOVI_VALUE exists in 2020/2021 source tables only (HVRI methodology).
sovi_val_frames = []
for vintage in [2020, 2021]:
    src_path = Path(f'../data/NRI_{vintage}.csv')
    if src_path.exists():
        s = pd.read_csv(src_path, usecols=['STCOFIPS', 'SOVI_VALUE'], low_memory=False)
        s['stcofips']    = s['STCOFIPS'].astype(str).str.zfill(5)
        s['nri_vintage'] = vintage
        s['sovi_value']  = s['SOVI_VALUE']
        sovi_val_frames.append(s[['stcofips', 'nri_vintage', 'sovi_value']])

if sovi_val_frames:
    sovi_val = pd.concat(sovi_val_frames, ignore_index=True)
    nri_long = nri_long.merge(sovi_val, on=['stcofips', 'nri_vintage'], how='left')
else:
    nri_long['sovi_value'] = np.nan

# Restrict to the three vintages we have ZHVI windows for
nri_long = nri_long[nri_long['nri_vintage'].isin([2020, 2021, 2023, 2025])].copy()

print(f'NRI vintage rows (3 vintages): {len(nri_long):,}')
print('sovi_value coverage by vintage:')
print(nri_long.groupby('nri_vintage')['sovi_value'].apply(lambda s: f'{(1-s.isna().mean()):.0%} populated'))

NRI vintage rows (3 vintages): 12,576
sovi_value coverage by vintage:
nri_vintage
2020    100% populated
2021    100% populated
2023      0% populated
2025      0% populated
Name: sovi_value, dtype: object


In [10]:
# Derive coastal flag from any vintage's CFLD_EALT > 0
# (NRI computes coastal-flooding EAL only for counties with coastline exposure -
#  Atlantic, Pacific, Gulf, Great Lakes - matching NOAA's coastal-shoreline definition)

def coastal_from_nri(path):
    df = pd.read_csv(path, usecols=['STCOFIPS', 'CFLD_EALT'], low_memory=False)
    df['stcofips'] = df['STCOFIPS'].astype(str).str.zfill(5)
    return set(df.loc[df['CFLD_EALT'] > 0, 'stcofips'].unique())

coastal_set = set()
for f in ['../data/NRI_2020.csv', '../data/NRI_2021.csv',
          '../data/NRI_2023.csv', '../data/NRI_2025.csv']:
    if Path(f).exists():
        coastal_set |= coastal_from_nri(f)

print(f'Coastal counties identified: {len(coastal_set)}')

Coastal counties identified: 533


In [11]:
# Join NRI vintage values to mean ZHVI on stcofips + nri_vintage
xsec = (
    nri_long
    .merge(zhvi_xsec, on=['stcofips', 'nri_vintage'], how='inner')
)

# Coastal flag
xsec['coastal'] = xsec['stcofips'].isin(coastal_set).astype(int)

# CDC-SVI regime indicator (NRI v1.19 released March 2023)
xsec['cdc_svi_regime'] = (xsec['nri_vintage'] >= 2023).astype(int)

# State fips, pop density (people per sq km; NRI 'area' is sq mi)
xsec['state']       = xsec['stcofips'].str[:2]
xsec['pop_density'] = xsec['population'] / (xsec['area'] * 2.58999)

# Sanity checks
assert xsec.duplicated(['stcofips', 'nri_vintage']).sum() == 0
assert xsec['zhvi'].gt(0).all()
assert xsec['eal_valt'].ge(0).all()
implied_risk = xsec['eal_valt'] * xsec['crf_value']
rel_err = ((xsec['risk_value'] - implied_risk).abs() / xsec['risk_value'].replace(0, np.nan)).fillna(0)
assert rel_err.max() < 1e-2, 'Risk = EAL * CRF identity failing'

print(f'Cross-section shape: {xsec.shape}')
print(f'Counties per vintage:')
print(xsec.groupby('nri_vintage').size())
print(f'\nCoastal counties in xsec: {xsec.loc[xsec.coastal == 1, "stcofips"].nunique()}')
xsec.head()

Cross-section shape: (12184, 24)
Counties per vintage:
nri_vintage
2020    3013
2021    3052
2023    3055
2025    3064
dtype: int64

Coastal counties in xsec: 458


,nri_id,stateabbrv,county,population,buildvalue,agrivalue,area,risk_value,eal_valt,sovi_score,...,stcofips,sovi_value,zhvi,n_months,window_start,window_end,coastal,cdc_svi_regime,state,pop_density
0,C01001,AL,Autauga,54571,5.075584e+09,21460000.0,594.448312,2.602012e+06,3.311627e+06,25.857312,...,01001,-3.17,193902.048732,12,2020-05-31,2021-04-30,0,0,01,35.444571
1,C01001,AL,Autauga,54571,5.075584e+09,21460000.0,594.448312,2.087423e+06,2.656700e+06,25.857312,...,01001,-3.17,213523.755668,12,2021-05-31,2022-04-30,0,0,01,35.444571
2,C01001,AL,Autauga,58764,9.123274e+09,24613998.0,610.470508,6.156054e+06,5.514047e+06,51.299999,...,01001,NaN,235452.153008,12,2022-09-30,2023-08-31,0,1,01,37.166236
3,C01001,AL,Autauga,58764,1.024141e+10,27630646.0,610.470508,2.051090e+07,1.975657e+07,38.040712,...,01001,NaN,256449.078997,10,2025-06-30,2026-03-31,0,1,01,37.166236
4,C01003,AL,Baldwin,182265,2.314295e+10,120383000.0,1589.815853,1.834143e+07,1.816858e+07,34.292471,...,01003,-1.03,270096.763822,12,2020-05-31,2021-04-30,1,0,01,44.264786


In [12]:
OUT_COLS = [
    'stcofips', 'state', 'stateabbrv', 'county',
    'nri_vintage', 'cdc_svi_regime',
    'window_start', 'window_end', 'n_months',
    'zhvi',
    'eal_valt', 'risk_value', 'crf_value',
    'resl_value', 'resl_score',
    'sovi_score', 'sovi_value',
    'population', 'area', 'pop_density',
    'coastal',
]

xsec_out = xsec[OUT_COLS].sort_values(['stcofips', 'nri_vintage']).reset_index(drop=True)

out_dir = Path('../data/processed')
xsec_out.to_csv(out_dir / 'hedonic_xsec.csv', index=False)
xsec_out.to_pickle(out_dir / 'hedonic_xsec.pkl')

print(f'Wrote {out_dir / "hedonic_xsec.csv"}')
print(f'Shape: {xsec_out.shape}')

Wrote ../data/processed/hedonic_xsec.csv
Shape: (12184, 21)
